In [5]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow_model_optimization as tfmot
import tensorflow as tf
import tf_keras as keras
from tensorflow.keras.datasets import mnist

# Load dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

# Build a simple model
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=(28, 28)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(10, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

# Apply pruning to the model
pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(initial_sparsity=0.0, final_sparsity=0.5, begin_step=0, end_step=1000)
}
pruned_model = tfmot.sparsity.keras.prune_low_magnitude(model, **pruning_params)

# Compile the pruned model
pruned_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the pruned model to finalize pruning
callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]
pruned_model.fit(x_train, y_train, epochs=2, validation_data=(x_test, y_test), callbacks=callbacks)

# Strip pruning wrappers to remove pruning-specific layers and metadata
pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

Epoch 1/5
1875/1875 [==============================] - 1s 509us/step - loss: 0.2946 - accuracy: 0.9146 - val_loss: 0.1388 - val_accuracy: 0.9598
Epoch 2/5
1875/1875 [==============================] - 1s 442us/step - loss: 0.1428 - accuracy: 0.9567 - val_loss: 0.0980 - val_accuracy: 0.9695
Epoch 3/5
1875/1875 [==============================] - 1s 450us/step - loss: 0.1060 - accuracy: 0.9680 - val_loss: 0.0848 - val_accuracy: 0.9740
Epoch 4/5
1875/1875 [==============================] - 1s 436us/step - loss: 0.0879 - accuracy: 0.9730 - val_loss: 0.0756 - val_accuracy: 0.9749
Epoch 5/5
1875/1875 [==============================] - 1s 465us/step - loss: 0.0734 - accuracy: 0.9769 - val_loss: 0.0722 - val_accuracy: 0.9779
Epoch 1/2
1875/1875 [==============================] - 2s 738us/step - loss: 0.0702 - accuracy: 0.9779 - val_loss: 0.0664 - val_accuracy: 0.9802
Epoch 2/2
1875/1875 [==============================] - 1s 538us/step - loss: 0.0564 - accuracy: 0.9825 - val_loss: 0.0652 - val_ac

In [6]:
# Convert the pruned model to a TensorFlow Lite quantized model
converter = tf.lite.TFLiteConverter.from_keras_model(pruned_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
quantized_model = converter.convert()

INFO:tensorflow:Assets written to: /var/folders/lr/1_ht819n4cl18qfz8jvz6fj80000gp/T/tmp194_t96r/assets


INFO:tensorflow:Assets written to: /var/folders/lr/1_ht819n4cl18qfz8jvz6fj80000gp/T/tmp194_t96r/assets
W0000 00:00:1781526206.084802 18564722 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1781526206.084813 18564722 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1781526206.085076 18564722 reader.cc:83] Reading SavedModel from: /var/folders/lr/1_ht819n4cl18qfz8jvz6fj80000gp/T/tmp194_t96r
I0000 00:00:1781526206.085391 18564722 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1781526206.085394 18564722 reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/lr/1_ht819n4cl18qfz8jvz6fj80000gp/T/tmp194_t96r
I0000 00:00:1781526206.086887 18564722 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1781526206.087084 18564722 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1781526206.092812 18564722 loader.cc:220] Running initialization op on SavedModel bundle

In [8]:
# Measure accuracy of the quantized model using the test set
from ai_edge_litert.interpreter import Interpreter as LiteRTInterpreter

interpreter = LiteRTInterpreter(model_content=quantized_model)
interpreter.allocate_tensors()

input_index = interpreter.get_input_details()[0]['index']
output_index = interpreter.get_output_details()[0]['index']

# Evaluate accuracy
correct_predictions = 0
for i in range(len(x_test)):
    input_data = x_test[i:i+1].astype('float32')
    interpreter.set_tensor(input_index, input_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_index)
    predicted_label = output.argmax()
    if predicted_label == y_test[i]:
        correct_predictions += 1

accuracy = correct_predictions / len(x_test)
print(f'Quantized model accuracy: {accuracy * 100:.2f}%')

Quantized model accuracy: 97.85%


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
